# Задание 1


In [1]:
from scipy.optimize import linprog
import numpy as np

In [2]:
cost = np.array([
    [2, 5, 3],
    [7, 7, 6]
])
stock = np.array([180, 220])
demand = np.array([110, 150, 140])
num_warehouse = 2
num_clients = 3

$x_{ij}$ - сколько забирается со i склада клиенту j  
$$f = \sum_{i,j} cost_{ij} * x_{ij}$$

In [3]:
c = cost.flatten()
print(c) 

[2 5 3 7 7 6]


Для каждого склада количество взятых предметов должно быть меньше, чем на складе:

$$\forall i: \sum_j x_{ij} \leq stock_i$$

In [8]:
A = []  # Создаём пустой список для будущей матрицы ограничений
b = []  # Создаём пустой список для правых частей (запасы складов)

# Цикл по каждому складу
for i in range(0, num_warehouse):
    # 1. Формируем строку матрицы A для i-го склада
    part_1 = [0] * (num_clients * i)            # Блок нулей перед "единичным" блоком
    print(part_1)
    part_2 = [1] * num_clients                  # Блок из единиц для клиентов данного склада
    print(part_2)
    part_3 = [0] * (num_clients * (num_warehouse - i - 1)) # Блок нулей после
    print(part_3)
    row = part_1 + part_2 + part_3              # Склеиваем все части в одну строку
    print(row)
    A.append(row)  # Добавляем строку в матрицу
    
    # 2. Добавляем соответствующее ограничение по запасам в вектор b
    b.append(stock[i])

# Преобразуем списки в массивы NumPy для эффективных вычислений
A = np.asarray(A)  # Теперь A — это двумерный массив (матрица)
b = np.asarray(b)  # Теперь b — это одномерный массив (вектор)

# Выводим результат
print("Матрица ограничений A (склады x переменных):")
print(A)
print("\nВектор правых частей b (запасы складов):")
print(b)

[]
[1, 1, 1]
[0, 0, 0]
[1, 1, 1, 0, 0, 0]
[0, 0, 0]
[1, 1, 1]
[]
[0, 0, 0, 1, 1, 1]
Матрица ограничений A (склады x переменных):
[[1 1 1 0 0 0]
 [0 0 0 1 1 1]]

Вектор правых частей b (запасы складов):
[180 220]


Для каждого клиента количество приобретаемых товаров должно быть больше на единицу, чем спрос:

$$\forall j: \sum_i x_{ij} \geq demand_j$$

Который также:

$$\forall j: - \sum_i x_{ij} \leq -demand_j$$

In [10]:
# Предыдущий код создал матрицу A и вектор b с ограничениями по складам
# Теперь преобразуем их обратно в списки, чтобы удобно добавлять новые строки
A = A.tolist()  # [[1,1,1,1,0,0,0,0,0,0,0,0], [0,0,0,0,1,1,1,1,0,0,0,0], ...]
b = b.tolist()  # [100, 150, 200]

# Цикл по каждому клиенту для добавления ограничений по спросу
for j in range(0, num_clients):
    # 1. Создаём шаблон для одного клиента: список длиной num_clients
    #    с -1 на позиции этого клиента и 0 в остальных
    #    Пример для клиента j=1 (второй клиент): [0, -1, 0, 0]
    client_pattern = [0] * j + [-1] + [0] * (num_clients - j - 1)

    # 2. "Размножаем" этот шаблон на все склады.
    #    Оператор * для списков создаёт повторение.
    #    Для j=1 и num_warehouse=3 получим: [0, -1, 0, 0, 0, -1, 0, 0, 0, -1, 0, 0]
    row_for_client = client_pattern * num_warehouse

    # 3. Добавляем эту строку в матрицу A
    A.append(row_for_client)

    # 4. Добавляем в вектор b отрицательный спрос этого клиента
    #    Отрицательное значение связано с тем, что в канонической форме
    #    все ограничения обычно записываются как A*x <= b.
    b.append(-demand[j])

# Преобразуем обновлённые списки обратно в массивы NumPy
A = np.asarray(A)  # Теперь матрица имеет (num_warehouse + num_clients) строк
b = np.asarray(b)  # Вектор соответствующей длины

print("Расширенная матрица ограничений A:")
print(A)
print("\nРасширенный вектор правых частей b:")
print(b)

Расширенная матрица ограничений A:
[[ 1  1  1  0  0  0]
 [ 0  0  0  1  1  1]
 [-1  0  0 -1  0  0]
 [ 0 -1  0  0 -1  0]
 [ 0  0 -1  0  0 -1]
 [-1  0  0 -1  0  0]
 [ 0 -1  0  0 -1  0]
 [ 0  0 -1  0  0 -1]]

Расширенный вектор правых частей b:
[ 180  220 -110 -150 -140 -110 -150 -140]


In [12]:
linprog(c=c, A_ub=A, b_ub=b)

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: 1900.0
              x: [ 1.100e+02  0.000e+00  7.000e+01  0.000e+00  1.500e+02
                   7.000e+01]
            nit: 5
          lower:  residual: [ 1.100e+02  0.000e+00  7.000e+01  0.000e+00
                              1.500e+02  7.000e+01]
                 marginals: [ 0.000e+00  1.000e+00  0.000e+00  2.000e+00
                              0.000e+00  0.000e+00]
          upper:  residual: [       inf        inf        inf        inf
                                    inf        inf]
                 marginals: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00
                              0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00
                              0.000e+00  0.000e+00  0.000e+00  0.000e+00]
                 margin

Ответ: 110 единиц со склада 1 клиенту 1, 70 единиц со склада 1 клиенту 3,
150 наименований со склада 2 клиенту 2, 70 наименований со склада 2 клиенту 3